In [2]:
from google.colab import files
files.upload()  # Upload kaggle.json


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"dobserine","key":"c2e4dbbb48001a136dbc27ad4ad9dfc6"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [4]:
!kaggle competitions download -c pink-ai-breast-cancer-risk-prediction-challenge


  0% 0.00/12.3M [00:00<?, ?B/s]
100% 12.3M/12.3M [00:00<00:00, 1.61GB/s]


In [5]:
!unzip pink-ai-breast-cancer-risk-prediction-challenge.zip -d ./pink_data


Archive:  pink-ai-breast-cancer-risk-prediction-challenge.zip
  inflating: ./pink_data/sample_submission.csv  
  inflating: ./pink_data/test.csv    
  inflating: ./pink_data/train.csv   


In [6]:
import pandas as pd

train = pd.read_csv("./pink_data/train.csv")
test = pd.read_csv("./pink_data/test.csv")
sample = pd.read_csv("./pink_data/sample_submission.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head()


Train shape: (893578, 14)
Test shape: (297860, 13)


,ID,feature_0,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,target,feature_12
0,1,2006,5.0,NaN,0.0,1.0,2.0,3.0,0.0,2.0,1.0,0.0,0.0,3.0
1,2,2013,10.0,2.0,0.0,1.0,2.0,3.0,0.0,2.0,3.0,0.0,0.0,6.0
2,3,2005,4.0,1.0,1.0,NaN,0.0,3.0,0.0,1.0,1.0,0.0,0.0,17.0
3,4,2007,7.0,3.0,0.0,NaN,3.0,4.0,1.0,2.0,1.0,1.0,1.0,1.0
4,5,2012,8.0,5.0,0.0,2.0,NaN,3.0,0.0,2.0,1.0,1.0,0.0,1.0


In [ ]:
# ================================
# 1️⃣ Imports
# ================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

# ================================
# 2️⃣ Load data
# ================================
train = pd.read_csv("./pink_data/train.csv")
test = pd.read_csv("./pink_data/test.csv")
sample = pd.read_csv("./pink_data/sample_submission.csv")

target = "target"
features = [c for c in train.columns if c not in ["ID", target]]

X = train[features]
y = train[target]
X_test = test[features]

# ================================
# 3️⃣ Scale features
# ================================
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

# ================================
# 4️⃣ Build MLP model
# ================================
model = Sequential([
    Dense(128, activation='relu', input_shape=(X.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  # Binary classification
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['AUC']
)

# ================================
# 5️⃣ Train model with early stopping
# ================================
early_stop = EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True)

history = model.fit(
    X, y,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=2
)

# ================================
# 6️⃣ Predict on test data
# ================================
y_test_preds = model.predict(X_test)

# ================================
# 7️⃣ Prepare submission
# ================================
submission = pd.DataFrame({
    "ID": test["ID"],
    "target": y_test_preds.flatten()
})
submission.to_csv("submission_mlp.csv", index=False)

# Optional: download in Colab
from google.colab import files
files.download("submission_mlp.csv")

print("\n✅ MLP submission ready!")


Epoch 1/100
22340/22340 - 72s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 2/100
22340/22340 - 73s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 3/100
22340/22340 - 66s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 4/100
22340/22340 - 66s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 5/100
22340/22340 - 66s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 6/100
22340/22340 - 65s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 7/100
22340/22340 - 65s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 8/100
22340/22340 - 74s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 9/100
22340/22340 - 72s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val_loss: nan
Epoch 10/100
22340/22340 - 73s - 3ms/step - AUC: 0.5000 - loss: nan - val_AUC: 0.5000 - val